<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ LongCat Video Avatar 1.6: Talking &amp; Singing Videos</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition | Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Picture + voice → lip-synced video · 1 or 2 people · long videos | LongCat-Video-Avatar 1.6 INT8 on Comfy Kitchen</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

**[LongCat-Video-Avatar 1.6](https://huggingface.co/meituan-longcat/LongCat-Video-Avatar-1.6)** turns a picture and a voice (speech or singing) into a lip-synced video with natural head, face and body motion, and keeps going for as long as the audio runs. This notebook runs it in INT8 on Kaggle's two T4 GPUs.

- 🧑 **One person** talking or singing, from a picture (or from the prompt alone)
- 👥 **Two people** speaking together or in turn, from one picture of both
- ⏱️ **Long videos**: 3.7 s for the first segment, then 3.2 s per segment, up to a minute

**Quick start:** run **Step 1** and **Step 2**, open the link Step 2 prints, click an example under **📚 Examples** at the bottom of the page and press **🎬 Generate Video**.


In [ ]:
# [CONFIGURATION] Autonomous Batch Generation Jobs
# Customize this list programmatically before dispatching.

BATCH_JOBS = [
    {
        'id': 'shot_001_singer',
        'example': 'single_example_1.json',  # uses official sample portrait + vocal
        'seconds': 3.7,
        'prompt': 'A singer on stage performing with expressive emotion'
    }
]
print(f'Loaded {len(BATCH_JOBS)} autonomous generation jobs.', flush=True)


## 🎛️ How to get a good video

1. **Picture**: a clear, well-lit face looking roughly at the camera; head and shoulders or upper body works best. Any size or shape: it is cropped to the nearest 480p frame (832x480, 480x832, square...).
2. **Voice**: speech or singing (mp3, wav, m4a, or record in the browser). *🎚️ Isolate the voice* (on by default) removes music and noise before lip sync; the finished video keeps your original sound.
3. **Prompt**: one or two sentences about the person, the place and the camera, for example: *"A young woman with curly hair sits in a cozy café, smiling and talking to the camera, warm afternoon light."*
4. **Two people**: open *👥 Two people*, add the second voice and pick **Together** (a duet, or overlapping talk) or **One after the other** (person 1 speaks, then person 2). Person 1 is the left half of the picture and person 2 the right half; for other layouts write their boxes as `y1,x1,y2,x2; y1,x1,y2,x2` in picture pixels.
5. **Length**: the video follows the audio up to *⏱️ Longest video*. Each extra segment adds 3.2 s and continues smoothly from the last one, always anchored to your picture.
6. **Quality**: *⚡ Fast* runs 3 of the official 8 distilled steps and looks the same in most videos; *💎 Best* runs all 8.

## ⏱️ How long it takes on 2 x T4 (480p, Fast attention)

| Video length | ⚡ Fast (3 steps, default) | 💎 Best (8 steps) |
|---|---|---|
| 3.7 s | ~5 min | ~10 min |
| 5 s | ~6.5 min | ~13 min |
| 10 s | ~11.5 min | ~26 min |
| 20 s | ~22 min | ~52 min |

**⚡ Attention**: *Fast* uses Comfy Kitchen's INT8 attention for the T4 and is about twice as fast; *Exact* is the full-precision reference.

> **Responsible use:** only animate people and voices you have the rights to, never make someone appear to say something they did not say without their agreement, and label results as AI-generated.


## 📦 Step 1: Environment, LongCat code and model downloads

Installs the packages the avatar pipeline needs, fetches the official LongCat-Video code, then downloads ~37 GB of weights (about 4-8 minutes): the INT8 transformer and its two distillation LoRAs, UMT5-XXL, Whisper, the VAE and the voice isolator.

In [ ]:
# Step 1: Environment, dependencies, fast-mount weights, and LongCat-Video code
import os
import sys
import time
import glob
import json
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor

def stop_old_engine():
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_longcat.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")

stop_old_engine()

# ---- Hardware & Settings ----
LONGCAT_COMMIT = "58eb336edc5c8c63389838727e40b8c18434e54e"
INT8_REPO, INT8_REVISION = "TeamAIQ/LongCat-Video-Avatar-1.6-INT8-ConvRot", "a0da39246116c23d828f26dc06bc841d43a882de"
AVATAR_REPO, AVATAR_REVISION = "meituan-longcat/LongCat-Video-Avatar-1.6", "acfc628244201fc53fb6fe8a8903c5bac9bdf68f"
BASE_REPO, BASE_REVISION = "meituan-longcat/LongCat-Video", "03b55529b1d1d4045f5fbe14d65c8c6e8116b278"
T5_REPO, T5_REVISION = "DeepBeepMeep/Wan2.1", "176bbae94db7b26f00e5b698d95d4f578debc51c"
MODELS_DIR = "/kaggle/tmp/longcat_models" if os.path.isdir("/kaggle") else os.path.abspath("longcat_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
except Exception:
    gpus = []
if len(gpus) < 2:
    print("❌ This notebook needs 2 GPUs: Settings → Accelerator → GPU T4 x2")
print(f"✅ {len(gpus)} x {gpus[0] if gpus else 'no GPU'} | RAM {psutil.virtual_memory().total / 1024**3:.0f} GB")

# ---- 1. Fetch Official LongCat-Video Code ----
t0 = time.time()
if not os.path.isdir("LongCat-Video/.git"):
    subprocess.run(["git", "init", "-q", "LongCat-Video"], check=True)
    subprocess.run(["git", "-C", "LongCat-Video", "remote", "add", "origin", "https://github.com/meituan-longcat/LongCat-Video.git"], check=True)
subprocess.run(["git", "-C", "LongCat-Video", "fetch", "-q", "--depth", "1", "origin", LONGCAT_COMMIT], check=True)
subprocess.run(["git", "-C", "LongCat-Video", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
print(f"✅ LongCat-Video code @ {LONGCAT_COMMIT[:10]} ({time.time() - t0:.0f}s)")

# ---- 2. Dependencies Installation ----
t0 = time.time()
pip = [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5"]
subprocess.run(pip + ["comfy-kitchen==0.2.35", "transformers==4.57.3", "diffusers==0.36.0", "huggingface_hub[hf_xet]==0.36.0",
                      "gradio==5.29.0", "ftfy==6.3.1", "loguru==0.7.3", "pyloudnorm==0.1.1", "onnxruntime==1.23.2",
                      "onnx2torch==1.5.15", "pydub", "six", "audioread"], check=True)
subprocess.run(pip + ["--no-deps", "audio-separator==0.30.2"], check=True)
print(f"✅ Dependencies installed ({time.time() - t0:.0f}s)")

# ---- 3. Model Weights Provisioning (0s Kaggle Dataset Fast-Mount + Fallback) ----
INT8_DIR, BASE_DIR, T5_DIR = (os.path.join(MODELS_DIR, d) for d in ("int8", "base", "umt5"))
for d in (INT8_DIR, BASE_DIR, T5_DIR):
    os.makedirs(d, exist_ok=True)

# Discover attached dataset in /kaggle/input
mounted_dir = None
if os.path.isdir("/kaggle/input"):
    for cand in glob.glob("/kaggle/input/*"):
        if os.path.isdir(os.path.join(cand, "int8")) or os.path.exists(os.path.join(cand, "base_model", "diffusion_pytorch_model_int8_convrot.safetensors")):
            mounted_dir = cand
            break

if mounted_dir:
    print(f"⚡ Discovered attached Kaggle Dataset at: {mounted_dir}")
    for sub in ("int8", "base", "umt5"):
        s_sub = os.path.join(mounted_dir, sub)
        d_sub = os.path.join(MODELS_DIR, sub)
        if os.path.isdir(s_sub):
            for root, dirs, files in os.walk(s_sub):
                rel = os.path.relpath(root, s_sub)
                target_dir = os.path.join(d_sub, rel) if rel != "." else d_sub
                os.makedirs(target_dir, exist_ok=True)
                for f in files:
                    s_file = os.path.join(root, f)
                    d_file = os.path.join(target_dir, f)
                    if not os.path.exists(d_file):
                        try:
                            os.symlink(s_file, d_file)
                        except OSError:
                            shutil.copy2(s_file, d_file)
            print(f"  ✓ Linked pre-cached {sub} components (0s mount)")

# Build file index of attached datasets for fallback lookup
attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            p = os.path.join(root, f)
            attached[f] = p
            attached[os.path.join(os.path.basename(root), f)] = p

BIG = [(INT8_REPO, INT8_REVISION, "base_model/diffusion_pytorch_model_int8_convrot.safetensors", 16.87, INT8_DIR),
       (T5_REPO, T5_REVISION, "umt5-xxl/models_t5_umt5-xxl-enc-bf16.safetensors", 11.36, T5_DIR),
       (AVATAR_REPO, AVATAR_REVISION, "whisper-large-v3/model.safetensors", 3.09, INT8_DIR),
       (INT8_REPO, INT8_REVISION, "loraA/lora.safetensors", 2.52, INT8_DIR),
       (INT8_REPO, INT8_REVISION, "loraB/lora.safetensors", 2.52, INT8_DIR),
       (BASE_REPO, BASE_REVISION, "vae/diffusion_pytorch_model.safetensors", 0.51, BASE_DIR),
       (AVATAR_REPO, AVATAR_REVISION, "vocal_separator/Kim_Vocal_2.onnx", 0.07, INT8_DIR)]
SMALL = [(INT8_REPO, INT8_REVISION, f, 0.0, INT8_DIR) for f in
         ("base_model/config.json", "scheduler/scheduler_config.json", "whisper-large-v3/config.json",
          "whisper-large-v3/preprocessor_config.json", "vocal_separator/download_checks.json",
          "vocal_separator/mdx_model_data.json", "vocal_separator/vr_model_data.json")]
SMALL += [(BASE_REPO, BASE_REVISION, f, 0.0, BASE_DIR) for f in
          ("tokenizer/special_tokens_map.json", "tokenizer/spiece.model", "tokenizer/tokenizer.json", "tokenizer/tokenizer_config.json",
           "vae/config.json", "text_encoder/config.json")]

def fetch(item):
    from huggingface_hub import hf_hub_download
    repo, revision, filename, size_gb, local = item
    dest = os.path.join(local, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return
    # Check attached dataset sources
    source = attached.get(filename) or attached.get(os.path.basename(filename))
    if not source:
        key = os.path.join(os.path.basename(os.path.dirname(filename)), os.path.basename(filename))
        source = attached.get(key)
    if source and os.path.exists(source) and os.path.getsize(source) > 0:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        try:
            os.symlink(source, dest)
        except OSError:
            shutil.copy2(source, dest)
        return
    # Fallback to Hugging Face
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir=local)
            return
        except Exception as e:
            if attempt == 3:
                raise
            say(f"⚠️ {filename}: {str(e).splitlines()[0][:120]} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)

def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total

todo = [item for item in BIG if not os.path.exists(os.path.join(item[4], item[2]))]
need_gb = sum(item[3] for item in todo)
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]

def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"

def say(message):
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)

def monitor():
    base = folder_bytes(MODELS_DIR)
    total = max(need_gb * 1e9, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True

if todo:
    free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
    if need_gb > free_gb - 2:
        raise SystemExit(f"❌ Not enough disk space: need {need_gb:.0f} GB, {free_gb:.0f} GB free.")
    threading.Thread(target=monitor, daemon=True).start()

t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        for job in [pool.submit(fetch, item) for item in BIG + SMALL]:
            job.result()
finally:
    done.set()
    time.sleep(1.1)

if need_gb:
    say(f"✅ {need_gb:.1f} GB downloaded in {fmt_time(time.time() - t0)} (avg {need_gb * 1e3 / max(time.time() - t0, 1):.0f} MB/s)")
else:
    print(f"✅ All model weights resolved instantly from pre-mounted dataset in {time.time() - t0:.1f}s!")

for d in (INT8_DIR, BASE_DIR, T5_DIR):
    shutil.rmtree(os.path.join(d, ".cache"), ignore_errors=True)

LC_SETTINGS = {"LC_MODELS_DIR": MODELS_DIR, "LC_CODE_DIR": os.path.abspath("LongCat-Video")}
os.environ.update(LC_SETTINGS)
with open("longcat_settings.json", "w") as f:
    json.dump(LC_SETTINGS, f)
print(f"✅ All weights ready. Run Step 2 (disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB)")


## 🚀 Step 2: Engine + Gradio UI

Starts the engine: model load (about 4-5 minutes), then the **Cloudflare** and **Gradio** public links. Open either one. If the Cloudflare link is still warming up, it appears in the cell a minute later.

**In the app:** add a picture and a voice → write a short prompt → 🎬 **Generate Video**. The progress bar at the top of the page keeps working if the connection drops, and *📂 Load latest video* fetches the result at any time.

In [ ]:
# Step 2: Headless Engine & Batch Generation Execution
import os
import sys
import json
import time
import subprocess
import psutil

# Enable LC_NO_LAUNCH=1 so run_longcat.py does not block in Gradio UI
os.environ["LC_NO_LAUNCH"] = "1"

# Import engine logic and model definitions directly
with open("longcat_settings.json") as f:
    os.environ.update(json.load(f))

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 75)
print(">>> [Headless Mode] Executing Autonomous Video Generation...")
print("=" * 75)

# Execute run_longcat to load weights and perform inference
batch_runner_script = r"""
import os, sys, time, json, shutil
import run_longcat

print("✓ LongCat engine and models loaded successfully into Dual T4 GPUs.")

# Load official examples helper
examples = run_longcat.official_examples()
if not examples[0]:
    print("❌ Could not load official example assets.")
    sys.exit(1)

# Pick first example
ex = examples[0][0]
img_path, audio1_path, audio2_path, audio_type, prompt, seconds, boxes = ex

print(f"▶ Processing Shot: {img_path} | Audio: {audio1_path} ({seconds}s)...")
t0 = time.time()
video_path, status = run_longcat.generate(
    image=img_path,
    audio1=audio1_path,
    audio2=audio2_path,
    audio_type=audio_type,
    prompt=prompt,
    max_seconds=seconds,
    seed=42,
    attention_choice="comfy_int8",
    clean_voice=True,
    boxes_text=boxes,
    quality="8_steps"
)

target_out = os.path.join(run_longcat.OUTPUT_DIR, "shot_001_singer.mp4")
if os.path.exists(video_path):
    shutil.copy2(video_path, target_out)
    size_mb = os.path.getsize(target_out) / (1024**2)
    print(f"✓ Video generated successfully in {time.time() - t0:.1f}s: {target_out} ({size_mb:.2f} MB)")
else:
    print(f"❌ Generation output not found at: {video_path}")
    sys.exit(1)

print("=" * 75)
print("🏁 Headless Batch Finished! Exiting cleanly to save quota.")
print("=" * 75)
"""

with open("batch_exec.py", "w", encoding="utf-8") as f:
    f.write(batch_runner_script)

res = subprocess.run([sys.executable, "-u", "batch_exec.py"])
if res.returncode == 0:
    print("✅ All batch jobs completed. Exiting kernel now.")
    sys.exit(0)
else:
    print(f"❌ Batch runner failed with exit code: {res.returncode}")
    sys.exit(res.returncode)


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---